# 01.07 · Under the hood: Runnables and LCEL (legacy-aware)

**Module 01 · Models, Messages & Prompts** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Lessons 01.01–01.06. |
| 🔑 Keys | `GROQ_API_KEY` (optional; almost everything here runs offline) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-11 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (fake model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** the `Runnable` protocol (`invoke`/`stream`/`batch`, async twins, `config`) shared by prompts, models, parsers, tools, agents and LangGraph graphs.
2. **Read and write** LCEL pipelines: `prompt | model | parser`, `RunnableLambda`, `RunnableParallel`, `RunnablePassthrough.assign`, and visualize them.
3. **Add resilience** with `with_fallbacks` and `with_retry`, and detect when a fallback fired.
4. **Build** a streaming transform with `RunnableGenerator` (the pattern used in LangChain's voice-agent tutorial).
5. **Recognize** legacy code (`LLMChain`, `AgentExecutor`, `ConversationChain`, ...), know it lives in `langchain-classic`, and know **when not to use LCEL** in v1.

## 📚 Official reading (verified 2026-10-11)

| Page | Read it for |
|---|---|
| [`Runnable` reference](https://reference.langchain.com/python/langchain-core/runnables/base/Runnable) | **Main page.** Every method: `invoke`, `batch`, `stream`, `astream_events`, `with_retry`, `with_fallbacks`, `with_config`, `get_graph` |
| [Runnables reference index](https://reference.langchain.com/python/langchain-core/runnables) | `RunnableLambda`, `RunnableParallel`, `RunnablePassthrough`, `RunnableGenerator`, ... |
| [What's new in LangChain v1 → `langchain-classic`](https://docs.langchain.com/oss/python/releases/langchain-v1) | What moved out of `langchain` and why |
| [Voice agent tutorial](https://docs.langchain.com/oss/python/langchain/voice-agent) | A modern pipeline of `RunnableGenerator`s (STT → agent → TTS) |

> **Why learn LCEL if v1 is agent-first?** Because (1) every model, tool, agent and graph *is* a Runnable, so this protocol is how you call all of them; (2) half the LangChain code on the internet uses `|` pipelines, and you need to read it; (3) for small linear transformations, LCEL is still the shortest correct code.

---
## Step 0 · Install and set up

`grandalf` is a small pure-Python graph-layout library. LangChain uses it to draw pipelines as ASCII art (`get_graph().draw_ascii()`).

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "grandalf>=0.8,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "grandalf",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/01-models-messages-prompts/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · One protocol for everything

A **`Runnable`** is anything with this interface (from `langchain_core.runnables`):

| Method | Sync / async | Meaning |
|---|---|---|
| `invoke(input, config)` | `ainvoke` | One input → one output |
| `batch(inputs, config)` | `abatch` | Many inputs → many outputs, in parallel |
| `stream(input, config)` | `astream` | One input → output in pieces |
| `astream_events(...)` | async | Typed events from every inner step |
| `with_config`, `with_retry`, `with_fallbacks` | – | Return a *new* Runnable with extra behavior |

Because every component speaks this protocol, components **compose**: the output of one becomes the input of the next.

In [ ]:
# 🧪 Offline: four very different objects, one interface.
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableLambda
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_messages([("system", "Answer in {style} style."), ("human", "{question}")])
model = ChatGroq(model=settings.model_fast, api_key="gsk_OFFLINE_INSPECTION_ONLY")   # never invoked
parser = StrOutputParser()
shout = RunnableLambda(lambda text: text.upper())

for name, obj in (("ChatPromptTemplate", prompt), ("ChatGroq", model), ("StrOutputParser", parser), ("RunnableLambda", shout)):
    print(f"{name:<19} is a Runnable: {isinstance(obj, Runnable)}")
print("\nprompt expects:", prompt.get_input_jsonschema()["required"])

### What just happened?

A prompt, a model, a parser and a plain function wrapped in `RunnableLambda` are all `Runnable`s. Each also describes its input schema (`get_input_jsonschema()`): the prompt needs `question` and `style`. In Module 02 you'll see that **tools** and **agents** are Runnables too, and in Module 07 LangGraph **graphs** are as well. Same `invoke`, same `stream`, same `config`.

---
## Part 2 · `|`: building a pipeline (LCEL)

**LCEL** (LangChain Expression Language) is just the `|` operator on Runnables. `a | b` creates a `RunnableSequence`: `b.invoke(a.invoke(x))`, and it streams and batches through every step automatically. We use a fake model so the whole section runs offline.

In [ ]:
# 🧪 Offline: prompt → model → parser. Each step's output is the next step's input.
from langchain.messages import AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake_model = GenericFakeChatModel(messages=iter([AIMessage("Runnables compose like Unix pipes."),
                                                 AIMessage("Streaming flows through every step.")]))
chain = prompt | fake_model | parser

print(type(chain).__name__, "with steps:", [type(step).__name__ for step in chain.steps])
print("invoke →", repr(chain.invoke({"style": "concise", "question": "What is LCEL?"})))
print("stream →", [piece for piece in chain.stream({"style": "concise", "question": "Does it stream?"})][:6], "...")

In [ ]:
# 🧪 Offline: draw the pipeline (needs grandalf from Step 0).
print(chain.get_graph().draw_ascii())

### What just happened?

- `chain` is a `RunnableSequence` of three steps. `invoke` returned a **string**, because `StrOutputParser` turns the `AIMessage` into its text.
- `stream` yielded string pieces: the parser streams too, so you didn't write any streaming code.
- The drawing shows the data flow `PromptInput → ChatPromptTemplate → GenericFakeChatModel → StrOutputParser → StrOutputParserOutput`. That's also the shape of the trace in LangSmith (Lesson 01.08).

---
## Part 3 · Functions, parallel branches, and passing data along

| Building block | What it does | Shortcut |
|---|---|---|
| `RunnableLambda(f)` | Wraps any function `f(input) -> output` | A plain function inside a pipeline is wrapped automatically |
| `RunnableParallel(a=r1, b=r2)` | Runs branches on the **same** input, returns `{"a": ..., "b": ...}` | A dict literal inside a pipeline |
| `RunnablePassthrough.assign(k=r)` | Keeps the input dict and **adds** key `k` | – |

In [ ]:
# 🧪 Offline: compute two things from the same input in parallel, then combine them.
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

stats = RunnableParallel(
    words=RunnableLambda(lambda text: len(text.split())),
    chars=RunnableLambda(len),
    shouting=shout,
)
print(stats.invoke("runnables are composable"))

enrich = RunnablePassthrough.assign(word_count=lambda d: len(d["question"].split()))
print(enrich.invoke({"question": "How do I read old LangChain code?", "style": "concise"}))

In [ ]:
# 🧪 Offline: a realistic mini-pipeline: add metadata, answer, and return both together.
fake_model = GenericFakeChatModel(messages=iter([AIMessage("Look for the | operator and follow the data.")]))
answer_with_meta = (
    RunnablePassthrough.assign(word_count=lambda d: len(d["question"].split()))
    | {"answer": prompt | fake_model | parser, "word_count": lambda d: d["word_count"]}   # dict → RunnableParallel
)
print(answer_with_meta.invoke({"question": "How do I read old LangChain code?", "style": "concise"}))

### What just happened?

- `RunnableParallel` ran three branches on the same string and returned a dict. Branches run **concurrently** (threads), which matters when they are model calls.
- `assign` kept `question` and `style` and added `word_count`.
- In the last pipeline, the **dict literal** was coerced into a `RunnableParallel` and the lambda into a `RunnableLambda`. This compact style is common in older tutorials. It's powerful but gets hard to read quickly: keep pipelines short.

---
## Part 4 · Resilience: `with_fallbacks` and `with_retry`

`primary.with_fallbacks([backup])` tries `primary`; if it raises (by default any `Exception`, configurable with `exceptions_to_handle`), it runs `backup` with the **same input**. Typical use: big model → smaller model, or provider A → provider B. Project 01's stretch goal uses `120b → 20b`.

How do you know whether the fallback fired? Make each branch label its output. With real models you can also read `response_metadata["model_name"]`.

In [ ]:
# 🧪 Offline: a primary that fails (simulated outage) and a backup that answers. Each labels its output.
def primary_call(question: str) -> dict:
    raise TimeoutError("primary model timed out (simulated)")

def backup_call(question: str) -> dict:
    return {"answered_by": "backup", "answer": f"short answer to {question!r}"}

resilient = RunnableLambda(primary_call).with_fallbacks(
    [RunnableLambda(backup_call)], exceptions_to_handle=(TimeoutError, ConnectionError),
)
print(resilient.invoke("what is a fallback?"))

In [ ]:
# 💸 Calls Groq (reasoning model → fast model fallback), ~200 tokens. The primary has an impossible timeout.
from mastery.models import get_model

if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    primary = get_model("reasoning", timeout=0.001, max_retries=0)        # Will time out.
    backup = get_model("fast", temperature=0, max_tokens=200, reasoning_effort="low")
    # Narrow the list: only provider errors (timeouts, connection, 5xx) fall back; bugs still raise.
    import groq
    reply = primary.with_fallbacks([backup], exceptions_to_handle=(groq.APIError,)).invoke("Name one benefit of fallbacks, in 10 words.")
    print("answered by:", reply.response_metadata.get("model_name"), "→", reply.text)

### What just happened?

- Offline: the primary raised `TimeoutError`, which is in `exceptions_to_handle`, so the backup answered: `answered_by: backup`. A `ValueError` would **not** be caught: narrow the exception list so real bugs aren't hidden behind a fallback.
- Live: `answered by: openai/gpt-oss-20b`, not 120b. Fallbacks are invisible to the caller unless you log which branch answered. Count them (Project 01 stretch).
- `with_retry` (Lesson 01.06) and `with_fallbacks` combine: retry the primary a couple of times, *then* fall back.

> ⚠️ **Production note: fallbacks that hide outages.**
> A fallback that fires silently can run your app on the weaker model for days. In production: emit a metric or trace tag every time a fallback fires, alert on the rate, test the fallback path regularly, and make sure the backup's output is validated just like the primary's. In agents, use model fallback middleware (Lesson 04.02).

---
## Part 5 · `RunnableGenerator`: streaming transforms

Some steps must work on a **stream**, not a finished value. Example from the [voice agent tutorial](https://docs.langchain.com/oss/python/langchain/voice-agent): text-to-speech should start speaking the first **sentence** while the model is still writing the rest. A `RunnableGenerator` wraps a function that takes an **iterator** of inputs and **yields** outputs, so it can buffer tokens and emit whole sentences.

In [ ]:
# 🧪 Offline: turn a token stream into a sentence stream (the first step of a speech pipeline).
from collections.abc import Iterator

from langchain_core.runnables import RunnableGenerator

SENTENCE_END = ".!?"

def to_sentences(tokens: Iterator[str]) -> Iterator[str]:
    """Buffer streamed text and yield one complete sentence at a time."""
    buffer = ""
    for token in tokens:
        buffer += token
        while any(mark in buffer for mark in SENTENCE_END):
            cut = min(buffer.find(mark) for mark in SENTENCE_END if mark in buffer) + 1
            yield buffer[:cut].strip()
            buffer = buffer[cut:]
    if buffer.strip():
        yield buffer.strip()   # Whatever is left when the stream ends.

In [ ]:
# 🧪 Offline: model → text → sentences. Each sentence arrives as soon as it is complete.
fake_model = GenericFakeChatModel(messages=iter([AIMessage("Hello there. Your order is confirmed! Anything else? Bye")]))
speech_ready = fake_model | StrOutputParser() | RunnableGenerator(to_sentences)

for sentence in speech_ready.stream("confirm my order"):
    print("🔊", sentence)

### What just happened?

The model streamed word-sized pieces; `to_sentences` buffered them and yielded **four** sentences, each as soon as its punctuation arrived (the last one when the stream ended). In the voice tutorial, three such generators are chained: audio → transcript → agent text → audio. You'll build a version in Module 13 (an elective).

Note: calling `.invoke()` on a pipeline ending in a generator *concatenates* the yielded strings ("Hello there.Your order..."), so generators are meant to be **streamed**.

---
## Part 6 · Reading legacy code: what moved to `langchain-classic`

LangChain v1 trimmed the `langchain` package to agent building blocks; older abstractions moved to the separate **`langchain-classic`** package ([v1 release notes](https://docs.langchain.com/oss/python/releases/langchain-v1)). Old tutorials break with import errors:

In [ ]:
# 🧪 Offline: the classic import path from 2023-era tutorials no longer exists in langchain 1.x.
try:
    from langchain.chains import LLMChain  # noqa: F401
except ModuleNotFoundError as error:
    print("ModuleNotFoundError:", error)
    print("→ It lives in `langchain-classic` now: from langchain_classic.chains import LLMChain")

How to translate what you find (the "modern" column is what this course teaches):

| Legacy code (pre-v1) | What it did | Modern v1 equivalent | Course lesson |
|---|---|---|---|
| `LLMChain(llm=..., prompt=...)` | Prompt + model | `prompt \| model` or just `model.invoke(messages)` | 01.03, 01.07 |
| `SimpleSequentialChain`, `SequentialChain` | Steps in order | `a \| b \| c`, or a LangGraph graph if there's state or branching | 01.07, 07.x |
| `ConversationChain` + `ConversationBufferMemory` | Chat with memory | `create_agent` + checkpointer (`thread_id`) | 03.x |
| `initialize_agent`, `AgentExecutor` | Tool-using agent loop | `create_agent` (built on LangGraph) | 02.x |
| `langgraph.prebuilt.create_react_agent` | Prebuilt ReAct agent | `langchain.agents.create_agent` | 02.x |
| `RetrievalQA`, `ConversationalRetrievalChain` | RAG | Retrieval tool + agent, or a 2-step RAG chain | 06.x |
| `.run()`, `.predict()`, `.__call__()` on chains | Old call styles | `.invoke()` | 01.01 |

Install `langchain-classic` **only** to run old code you can't rewrite yet. New code shouldn't depend on it.

---
## Part 7 · When (not) to use LCEL in v1

| Use LCEL `\|` for... | Use something else for... |
|---|---|
| Short, linear, stateless transforms: prompt → model → parser | **Agents** (a model deciding which tools to call in a loop): `create_agent` (Module 02) |
| Streaming transforms (`RunnableGenerator`) | **Branching, loops, retries with state, human approval**: LangGraph (Modules 03, 07, 08) |
| Quick parallel fan-out (`RunnableParallel`) | **Long-running, resumable work**: LangGraph with a checkpointer |
| Reading other people's code | Anything you'd struggle to explain from the ASCII drawing |

The test: an `if` or a loop alone is ordinary Python (write a function). Reach for LangGraph when you need state across steps, persistence, interrupts for a human, or durable execution, not a longer pipe.

---
## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain.chains'` | Legacy tutorial on langchain 1.x | Rewrite with the table above, or `pip install langchain-classic` temporarily |
| `ImportError: Install grandalf to draw graphs` | `draw_ascii()` without `grandalf` | Step 0, or use `chain.get_graph().draw_mermaid()` |
| `TypeError` about input types in the middle of a chain | A step returns the wrong shape for the next (e.g. an `AIMessage` where a dict is expected) | Draw the graph; test each step with `.invoke()` alone |
| Fallback "never fires" | The raised exception isn't in `exceptions_to_handle` | List the provider's timeout/connection/5xx errors, or check with `explain_model_error` |
| Generator chain `.invoke()` returns glued text | Generators are for streaming | Use `.stream()` |
| Unreadable one-line pipelines with nested dicts and lambdas | LCEL overuse | Name the steps; move logic into functions or a LangGraph node |

## ⚠️ Production notes

> ⚠️ **Production note: lambdas in pipelines.**
> Anonymous lambdas show up as `RunnableLambda` with an unhelpful name in traces and can't be unit-tested on their own. In production: use named functions (or `.with_config(run_name="...")`), give each step a docstring, and unit-test steps in isolation with fake models, exactly as this notebook does.

## 🏋️ Exercises

1. **(Easy)** Add a fourth step to `chain` that counts words: `chain | RunnableLambda(lambda s: len(s.split()))`. Draw the new graph.
2. **(Easy)** Rewrite this legacy snippet in v1 style: `chain = LLMChain(llm=llm, prompt=prompt); chain.run(style="pirate", question="hi")`.
3. **(Medium)** Build a `RunnableParallel` that asks the same question to `get_model("fast")` and `get_model("reasoning")` at once and returns both texts (with a key). Compare wall time with two sequential `invoke`s.
4. **(Stretch)** Make `to_sentences` smarter: don't split on the dot in numbers like `3.5` or abbreviations like `e.g.`. Write three small asserts that prove it works.

<details>
<summary><b>Show solutions</b></summary>

**2.**
```python
chain = prompt | llm | StrOutputParser()
chain.invoke({"style": "pirate", "question": "hi"})
# Or, without LCEL: llm.invoke(prompt.invoke({"style": "pirate", "question": "hi"})).text
```

**3.**
```python
if HAVE_GROQ_KEY:
    both = RunnableParallel(
        fast=get_model("fast", reasoning_effort="low") | StrOutputParser(),
        reasoning=get_model("reasoning", reasoning_effort="low") | StrOutputParser(),
    )
    print(both.invoke("One sentence: what is LCEL?"))
```
Wall time ≈ the slower of the two, not the sum.

**4.** One approach: only treat `.` as an end when the next character is a space or the end of the stream, and the previous character isn't a digit. Then `assert list(to_sentences(iter(["It costs 3.5 dollars. Ok."]))) == ["It costs 3.5 dollars.", "Ok."]`. This rule handles `3.5` but still splits after `e.g.` (a space follows), and across chunks it must hold back a trailing `.` until the next chunk shows what follows. For abbreviations, keep a small set (`{"e.g.", "i.e.", "etc."}`) and check the last word before splitting; real sentence splitting is a solved-but-messy problem (look at `pysbd` or `nltk`'s Punkt), which is why a streaming UI usually just renders tokens.
</details>

## ✅ Checkpoint

- [ ] I can explain why prompts, models, parsers, tools, agents and graphs share `invoke`/`stream`/`batch`.
- [ ] I built and drew a `prompt | model | parser` pipeline, and used `RunnableParallel` and `assign`.
- [ ] I used `with_fallbacks` with a narrow exception list and could tell which branch answered.
- [ ] I built a sentence-streaming `RunnableGenerator`.
- [ ] I can translate `LLMChain` / `AgentExecutor` / `ConversationChain` code into v1 equivalents.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Docs drill (5 min)

Where does the LangChain v1 migration guide talk about `langchain-classic`, and what does it say should move there?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Docs: *Migrate → LangChain v1* (`/oss/python/migrate/langchain-v1`), section on `langchain-classic`.
2. It lists the legacy chains, retrievers, indexing API and hub that moved out of `langchain` into `langchain-classic`.
3. Then check `langchain-classic` on PyPI (release history: still released?) and search GitHub issues for `LLMChain` migration questions. Log the dates.

</details>

## 🔎 Research checkpoint

Find one LangChain tutorial or blog post from before v1 (2023–2025) that uses `LLMChain`, `AgentExecutor` or `ConversationChain`. Translate its core into v1 using the Part 6 table, and log the source, which imports broke, and your translation in [`docs/research-log.md`](../../docs/research-log.md). Check the [v1 release notes](https://docs.langchain.com/oss/python/releases/langchain-v1) for anything the table missed.

## Summary

- `Runnable` is the common protocol; `|` (LCEL) composes Runnables into sequences that stream and batch for free.
- `RunnableLambda`, `RunnableParallel`, `RunnablePassthrough.assign` and `RunnableGenerator` cover most pipeline needs.
- `with_fallbacks` / `with_retry` add resilience; always know when a fallback fired.
- Legacy chains live in `langchain-classic`. In v1: `create_agent` for agents, LangGraph for state and control flow, LCEL for short linear transforms.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 01.08 · First look at LangSmith tracing** → [`01-08-first-traces.ipynb`](01-08-first-traces.ipynb).

[Module 01 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)